# Option 3 (Regularized): Native CNN with anti-overfitting changes

This is a regularized variant of the from-scratch native CNN. Compared to the baseline (which reached ~0.98 train vs ~0.85 validation accuracy — a clear overfitting gap), it adds several changes aimed at closing that gap:

- **Stronger augmentation** — `RandomFlip`, wider `RandomRotation`, plus `RandomZoom` and `RandomContrast`.
- **Reduced model capacity** — block widths halved (entry 64; residual blocks 128 → 256 → 512; head 512) so the network has fewer parameters to memorize the small (~2,400-image) dataset.
- **More dropout** — `SpatialDropout2D` inside every residual block, plus heavier head dropout (0.4).
- **Decoupled weight decay** — the optimizer is `AdamW` with `weight_decay=1e-4`, a cleaner form of L2 regularization than adding a penalty to the loss.
- **Label smoothing** — `CategoricalCrossentropy(label_smoothing=0.1)`. Note this raises the loss floor (~0.17 for 3 classes), so we no longer chase a training loss below 0.1 — generalization is the goal now.

Everything else (data pipeline, cosine LR schedule, softmax output, one-hot labels, `val_loss`-based checkpointing and early stopping) matches the baseline notebook.


In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from pathlib import Path


In [ ]:
image_size = (128, 128)
batch_size = 32
seed = 42

# Resolve the dataset path whether the notebook runs from the repo root or Assignment_2.
dataset_dir = Path("Assignment_2/ImageDataset/images")
if not dataset_dir.exists():
    dataset_dir = Path("ImageDataset/images")

image_dataset = tf.keras.utils.image_dataset_from_directory(
    str(dataset_dir),
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed,
)

class_names = image_dataset.class_names
image_batches = []
label_batches = []

# Keep images and labels from the same batch so they stay correctly matched.
for images, labels in image_dataset:
    image_batches.append(images.numpy())
    label_batches.append(labels.numpy())

x_data = np.concatenate(image_batches, axis=0)
y_data = np.concatenate(label_batches, axis=0)

train_images, test_images, train_labels, test_labels = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=seed,
    stratify=y_data,
)

(x_train, y_train), (x_test, y_test) = (train_images, train_labels), (test_images, test_labels)

num_classes = len(class_names)

# No pretrained backbone here, so we just scale raw pixels to the 0..1 range.
image_train = x_train.astype("float32") / 255.0
image_test = x_test.astype("float32") / 255.0

# categorical_crossentropy expects one-hot encoded labels rather than integer class ids.
labels_train = to_categorical(y_train, num_classes=num_classes)
labels_test = to_categorical(y_test, num_classes=num_classes)

print(class_names)
print(image_train.shape, labels_train.shape)
print(image_test.shape, labels_test.shape)
print(image_train.dtype, image_train.min(), image_train.max())
print(image_test.dtype, image_test.min(), image_test.max())


In [ ]:
input_shape = image_train.shape[1:]

# Stronger, still label-preserving augmentation to force more input variety and
# reduce overfitting.
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.10),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

inputs = Input(shape=input_shape, name="input_image")
x = data_augmentation(inputs)

# Entry block: halved width (128 -> 64) to shrink capacity for this small dataset.
x = tf.keras.layers.Conv2D(64, 3, strides=2, padding="same", name="entry_conv")(x)
x = tf.keras.layers.BatchNormalization(name="entry_batchnorm")(x)
x = tf.keras.layers.Activation("relu", name="entry_relu")(x)

previous_block_activation = x  # Source tensor for the first residual shortcut.

# Residual blocks with separable convolutions (a compact "mini-Xception").
# Widths halved vs the baseline ([256, 512, 728] -> [128, 256, 512]) and a
# SpatialDropout2D added per block to further regularize the feature maps.
for block_index, size in enumerate([128, 256, 512], start=1):
    x = tf.keras.layers.Activation("relu", name=f"block{block_index}_relu1")(x)
    x = tf.keras.layers.SeparableConv2D(size, 3, padding="same", name=f"block{block_index}_sepconv1")(x)
    x = tf.keras.layers.BatchNormalization(name=f"block{block_index}_batchnorm1")(x)

    x = tf.keras.layers.Activation("relu", name=f"block{block_index}_relu2")(x)
    x = tf.keras.layers.SeparableConv2D(size, 3, padding="same", name=f"block{block_index}_sepconv2")(x)
    x = tf.keras.layers.BatchNormalization(name=f"block{block_index}_batchnorm2")(x)

    x = tf.keras.layers.MaxPooling2D(3, strides=2, padding="same", name=f"block{block_index}_pool")(x)

    # Project the shortcut to the new shape, then add it back to the main path.
    residual = tf.keras.layers.Conv2D(
        size, 1, strides=2, padding="same", name=f"block{block_index}_residual"
    )(previous_block_activation)
    x = tf.keras.layers.add([x, residual], name=f"block{block_index}_add")

    # Spatial dropout drops whole feature-map channels, a stronger regularizer
    # than element-wise dropout for convolutional activations.
    x = tf.keras.layers.SpatialDropout2D(0.1, name=f"block{block_index}_spatial_dropout")(x)
    previous_block_activation = x

x = tf.keras.layers.SeparableConv2D(512, 3, padding="same", name="head_sepconv")(x)
x = tf.keras.layers.BatchNormalization(name="head_batchnorm")(x)
x = tf.keras.layers.Activation("relu", name="head_relu")(x)

x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
# Heavier dropout at the classifier head to further curb overfitting.
x = tf.keras.layers.Dropout(0.4, name="dropout_regularization")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)

model = Model(inputs=inputs, outputs=outputs, name="animal_native_cnn_regularized")

# Cosine-decayed learning rate: the same smooth schedule used in the baseline.
epochs = 60
steps_per_epoch = int(np.ceil(len(image_train) / batch_size))
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=steps_per_epoch * epochs,
    alpha=0.0,
)

# AdamW applies decoupled weight decay (a cleaner form of L2 regularization),
# and label smoothing softens the one-hot targets. Both reduce overfitting at
# the cost of a higher loss floor, so the training loss will not (and should
# not) fall below ~0.1 here.
model.compile(
    optimizer=AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

model.summary()


In [ ]:
checkpoint_dir = Path("Assignment_2")
if not checkpoint_dir.exists():
    checkpoint_dir = Path(".")

best_model_path = checkpoint_dir / "animal_native_cnn_regularized_best.keras"

# Both callbacks track val_loss, the metric we actually care about for
# generalization. EarlyStopping restores the best-val weights when it triggers.
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=12,
        restore_best_weights=True,
    ),
]

history = model.fit(
    image_train,
    labels_train,
    validation_data=(image_test, labels_test),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
import matplotlib.pyplot as plt

# Pull the per-epoch metrics recorded during training.
history_metrics = history.history
epoch_range = range(1, len(history_metrics["loss"]) + 1)

fig, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize=(14, 5))

# Left: loss curves. The 0.1 line is now just a reference; label smoothing puts
# a floor near ~0.17, so we no longer expect the training loss to reach it.
loss_axis.plot(epoch_range, history_metrics["loss"], label="Training loss")
loss_axis.plot(epoch_range, history_metrics["val_loss"], label="Validation loss")
loss_axis.axhline(0.1, color="gray", linestyle="--", linewidth=1, label="Reference loss = 0.1")
loss_axis.set_title("Loss over epochs")
loss_axis.set_xlabel("Epoch")
loss_axis.set_ylabel("Loss")
loss_axis.legend()
loss_axis.grid(True, alpha=0.3)

# Right: accuracy curves (the "learning" progress).
accuracy_axis.plot(epoch_range, history_metrics["accuracy"], label="Training accuracy")
accuracy_axis.plot(epoch_range, history_metrics["val_accuracy"], label="Validation accuracy")
accuracy_axis.set_title("Accuracy over epochs")
accuracy_axis.set_xlabel("Epoch")
accuracy_axis.set_ylabel("Accuracy")
accuracy_axis.legend()
accuracy_axis.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = int(np.argmin(history_metrics["val_loss"])) + 1
train_val_gap = max(history_metrics["accuracy"]) - max(history_metrics["val_accuracy"])
print(f"Lowest training loss:   {min(history_metrics['loss']):.4f}")
print(f"Lowest validation loss: {min(history_metrics['val_loss']):.4f} (epoch {best_epoch})")
print(f"Best training accuracy:   {max(history_metrics['accuracy']):.4f}")
print(f"Best validation accuracy: {max(history_metrics['val_accuracy']):.4f}")
print(f"Train/val accuracy gap:   {train_val_gap:.4f}  (smaller is better)")


## What to look for

Compared to the baseline run, these regularization changes should:

- **Narrow the train/validation gap** — training accuracy will likely be lower than the baseline's ~0.98 (that is expected and healthy), while validation accuracy should hold near or above ~0.85 with a smoother, less spiky validation-loss curve.
- **Keep training loss above ~0.1** — label smoothing puts a floor on the loss, so the dashed "Reference loss = 0.1" line is now just a baseline reference, not a goal.

If validation accuracy is still much lower than training accuracy, the next levers are: even smaller block widths, fewer residual blocks, or more data.


In [ ]:
model_dir = Path("Assignment_2")
if not model_dir.exists():
    model_dir = Path(".")

model_path = model_dir / "animal_native_cnn_regularized.keras"
weights_path = model_dir / "animal_native_cnn_regularized.weights.h5"
labels_path = model_dir / "class_names_native_cnn_regularized.json"

model.save(model_path)
model.save_weights(weights_path)

with open(labels_path, "w", encoding="utf-8") as labels_file:
    json.dump(class_names, labels_file, indent=2)

print(f"Saved restored best model to: {model_path.resolve()}")
print(f"Saved checkpointed best model to: {best_model_path.resolve()}")
print(f"Saved weights to: {weights_path.resolve()}")
print(f"Saved class labels to: {labels_path.resolve()}")


In [ ]:
import matplotlib.pyplot as plt

loaded_model = tf.keras.models.load_model(model_path)

with open(labels_path, "r", encoding="utf-8") as labels_file:
    saved_class_names = json.load(labels_file)

loaded_loss, loaded_accuracy = loaded_model.evaluate(image_test, labels_test, verbose=0)
print(f"Loaded model test loss: {loaded_loss:.4f}")
print(f"Loaded model test accuracy: {loaded_accuracy:.4f}")

sample_count = min(9, len(image_test))
sample_indices = np.arange(sample_count)
predictions = loaded_model.predict(image_test[sample_indices], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(labels_test, axis=1)

fig, axes = plt.subplots(3, 3, figsize=(10, 9), constrained_layout=True)
for axis, image_index, prediction_index in zip(axes.ravel(), sample_indices, range(sample_count)):
    true_label = saved_class_names[true_labels[image_index]]
    predicted_label = saved_class_names[predicted_labels[prediction_index]]
    confidence = predictions[prediction_index][predicted_labels[prediction_index]]
    title_color = "green" if true_label == predicted_label else "red"

    axis.imshow(x_test[image_index].astype("uint8"))
    axis.set_title(
        f"True: {true_label}\nPred: {predicted_label} ({confidence:.2f})",
        color=title_color,
        fontsize=10,
    )
    axis.axis("off")

plt.show()
